In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pickle

# Notebook description

* This Notebook generates simulated price/temperature data to use in the thesis.
* We will discuss with the student how many days (n_days) and horizons (n_horizons) are required.
* The simulated results are stored in a dictionary (sims_dict)
* For conveniance, after generating the data the student can save it in a pickle file useing the code snippet below.

In [ ]:

np.random.seed(42)

# ============================================================
# 1. SETUP
# ============================================================

n_days = 10
n_paths = 5000
n_horizons = 8760  # 365*24

asof_dates = pd.date_range("2020-01-01", periods=n_days, freq="D")
hours = np.arange(1, n_horizons + 1)
path_cols = [f"path_{i}" for i in range(1, n_paths + 1)]

series_names = ["price", "temp"]

sims_dict = {}
realized_dict = {s: pd.DataFrame(index=asof_dates, columns=hours, dtype=float)
                 for s in series_names}

# ============================================================
# 2. FAKE JOINT SIMULATIONS (price & temperature)
# ============================================================

base_price = 50.0
amp_price_day = 10.0          # intraday amplitude
amp_price_year = 5.0          # yearly amplitude (keep modest)

base_temp = 10.0
amp_temp_day = 8.0            # intraday amplitude
amp_temp_year = 12.0          # yearly amplitude (bigger seasonal swing)

sigma_price = 5.0
sigma_temp = 3.0
rho = 0.5

cov = np.array([
    [sigma_price**2, rho * sigma_price * sigma_temp],
    [rho * sigma_price * sigma_temp, sigma_temp**2]
])

# Precompute Cholesky for fast correlated shocks
L = np.linalg.cholesky(cov)  # cov = L @ L.T

for asof in asof_dates:
    # Build an hourly datetime index for the horizon starting at asof
    dt_index = pd.date_range(asof, periods=n_horizons, freq="h")

    # Intraday seasonality (hour of day)
    hod = dt_index.hour.values  # 0..23
    day_phase = 2 * np.pi * hod / 24.0

    # NEW: Yearly seasonality (day of year)
    doy = dt_index.dayofyear.values  # 1..365/366
    year_phase = 2 * np.pi * (doy - 1) / 365.0  # keep 365 for simplicity

    # Hourly means (shape: n_horizons,)
    mean_price = (
    base_price
    + amp_price_day * (
        0.8 * np.sin(day_phase - np.pi) +
        0.2 * np.sin(2 * (day_phase - np.pi))
    )
    + amp_price_year * np.cos(year_phase)
    )

    mean_temp = (
    base_temp
    - amp_temp_day * np.cos(2 * np.pi * (hod - 5) / 24.0)
    + amp_temp_year * np.cos(year_phase + np.pi)
    )

    # Draw all shocks at once: Z ~ N(0, I), then eps = Z @ L.T
    Z = np.random.normal(size=(n_horizons, n_paths, 2))            # iid
    eps = Z @ L.T                                                 # correlated

    sims_price = mean_price[:, None] + eps[:, :, 0]
    sims_temp  = mean_temp[:, None]  + eps[:, :, 1]

    # Realized values: one draw per hour (same distribution)
    Z_real = np.random.normal(size=(n_horizons, 2))
    eps_real = Z_real @ L.T
    realized_price = mean_price + eps_real[:, 0]
    realized_temp  = mean_temp  + eps_real[:, 1]

    realized_dict["price"].loc[asof, :] = realized_price
    realized_dict["temp"].loc[asof, :]  = realized_temp

    df_price = pd.DataFrame(sims_price, index=hours, columns=path_cols)
    df_temp  = pd.DataFrame(sims_temp,  index=hours, columns=path_cols)

    sims_dict[asof] = {"price": df_price, "temp": df_temp}


In [ ]:
#Example output: Temperature
sims_dict[pd.Timestamp("2020-01-07 00:00:00")]['temp'].head()

,path_1,path_2,path_3,path_4,path_5,path_6,path_7,path_8,path_9,path_10,...,path_4991,path_4992,path_4993,path_4994,path_4995,path_4996,path_4997,path_4998,path_4999,path_5000
1,-3.568244,-8.183897,-9.214628,-3.178439,-3.140655,-1.338648,-1.489545,-3.235099,-5.350827,0.174550,...,-5.124930,-6.536630,-0.918007,-1.439845,-1.376083,-4.588840,-8.910377,-3.947807,-2.859155,-5.908194
2,-8.451151,-6.098827,-8.302976,-5.875975,-7.153809,-6.915484,-0.420659,-1.708636,-4.974505,-6.161091,...,-6.204185,0.205886,-3.226020,-5.266871,-2.889801,-6.818074,-8.447160,-11.476225,-4.556294,-6.898224
3,-4.496128,-9.297947,-8.547708,-1.704059,-9.099432,-12.557814,-5.673786,-6.447632,-12.246192,-10.010846,...,-11.045049,-4.394279,-11.217621,-2.807756,-5.606037,-6.843713,-9.594270,-8.919512,-9.452450,-8.990890
4,-5.106521,-9.178193,-7.314100,-9.761696,-8.847755,-9.556966,-5.206145,-7.018488,-5.009038,-12.632322,...,-6.859669,-5.399227,-7.853777,-14.009560,-13.740231,-5.412662,-2.812334,-8.334312,-8.671430,-7.294398
5,-10.291873,-14.705944,-8.108292,-11.335669,-10.012579,-10.703640,-10.743967,-10.031929,-9.359866,-10.014458,...,-8.364665,-11.319280,-8.857444,-12.048141,-8.510199,-11.652383,-11.667781,-6.015583,-11.118452,-6.654643


In [ ]:
#Example output: Price
sims_dict[pd.Timestamp("2020-01-07 00:00:00")]['price'].head()

,path_1,path_2,path_3,path_4,path_5,path_6,path_7,path_8,path_9,path_10,...,path_4991,path_4992,path_4993,path_4994,path_4995,path_4996,path_4997,path_4998,path_4999,path_5000
1,46.448618,51.012391,60.752460,53.341281,60.978037,54.946704,54.208876,52.133410,53.819922,57.876589,...,57.551873,51.169590,54.782222,54.059019,60.912426,54.295947,42.513878,57.672405,54.518490,48.036010
2,50.945890,57.733971,50.377650,61.417409,55.110173,53.601147,58.654204,60.014233,55.185901,49.255514,...,48.981434,62.699516,50.366401,53.684035,54.390181,47.624461,53.004389,49.356473,52.673491,48.892383
3,61.462291,55.919885,46.544666,59.669808,58.546714,47.511508,55.444676,53.180720,46.708371,58.254284,...,55.097387,62.086663,50.651753,57.369024,54.592716,52.535353,53.347886,61.981816,64.182567,50.886558
4,56.294170,55.603722,52.948425,53.323580,53.221432,54.208097,58.860389,52.322421,51.135493,46.616496,...,56.536491,56.688836,51.996165,44.895992,47.387089,54.906475,59.888057,54.354688,46.639353,55.052217
5,42.791423,50.847934,49.451159,49.554475,53.116736,40.146625,55.322904,41.008493,49.784545,45.844339,...,45.647621,49.542222,43.497471,53.007709,48.979449,48.531028,54.409157,48.819987,44.939866,52.459924


**Save the simulated data**

In [ ]:
# output_path = "sims_dict.pkl"

# with open(output_path, "wb") as f:
#     pickle.dump(sims_dict, f, protocol=pickle.HIGHEST_PROTOCOL)
    

**Re-load the saved data**

In [ ]:
# with open("sims_dict.pkl", "rb") as f:
#     sims_dict_loaded = pickle.load(f)